## PURPOSE OF THIS NOTEBOOK: 

The goal is to create a pitstop calculator based on telemetry data. It will report: 
1. average fuel consumption 
2. fuel consumption needed to add a lap before pitstop
3. total time estimated from per lap fuel saved
4. laptime expected for level of fuel save

### Factors I would need to calculate to complete this 
1. rate of acceleration

### NOTES: 
1. Data was collected varying frequencies. Given we know the frequency cycle rates, the formula to conver to seconds will be (F = 1/T) 
2. Tables labeled as Events report the event in seconds 
3. I need to think of a way to calculate fuel used when additional fuel is put in during a pitstop. First I need to isolate the lap where
4. Cars of all classes in LMU all have 100L liters of fuel. From the limited testing I've done the fuel state reported in telemetry is the liters of fuel left, not virtual energy

## Step 01: Explore Data

In [18]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

In [2]:
# import the data we need 

# Lap, this has the time in seconds where the lap started as well as the lap number, will be used as a primary indicator for position 
lap = pd.read_csv("data/Lap.csv")

# Fuel Level, the fuel level needed, Frequency is 20
fuel_level = pd.read_csv("data/Fuel Level.csv")

# In Pits, table that logs the timestamp I enter the pits. Useful to find pit in laps to remove from timing analysis
pit_indicator = pd.read_csv("data/In Pits.csv")

In [3]:
fuel_level.shape

(50908, 1)

In [4]:
fuel_level

,value
0,92.000000
1,92.000000
2,92.000000
3,92.000000
4,92.000000
...,...
50903,16.962720
50904,16.962633
50905,16.962545
50906,16.962456


In [5]:
# Create new row, ts for fuel_level, will represent the amount of time in seconds. 
# we know the frequency is 20, which means we get a reading every .05 seconds (using the frequency equation)
fuel_level['ts'] = .05 * fuel_level.index

# renaming fuel level for merge
fuel_level.rename({"value" : "fuel_level"}, inplace=True, axis=1)

In [6]:
fuel_level

,fuel_level,ts
0,92.000000,0.00
1,92.000000,0.05
2,92.000000,0.10
3,92.000000,0.15
4,92.000000,0.20
...,...,...
50903,16.962720,2545.15
50904,16.962633,2545.20
50905,16.962545,2545.25
50906,16.962456,2545.30


In [7]:
lap

,ts,value
0,14.6425,0
1,238.5600,1
2,345.8400,2
3,452.9800,3
4,559.9200,4
5,665.9000,5
6,773.0600,6
7,879.4600,7
8,985.8000,8
9,1093.9200,9


### Goal: 

I want to add the lap numbers to the fuel level to know which lap is which.
I calculated the amount of time passed, now I need to make every row where the time is less than the lap assciocated with the other, give it the lap it's in

In [8]:
# Create a new column, lap, based on the amount of time passed using the lap table as a LUT
# I want to 
def lookup_up_lap(times):
    '''
    Add lap number based on time passed
    '''
    time_index = times <= lap['ts']
    lap_num = lap['value'][time_index]
    return lap_num.values[0]

fuel_level['LapNum'] = fuel_level['ts'].apply(lookup_up_lap)

In [9]:
pit_indicator['LapNum'] = pit_indicator['ts'].apply(lookup_up_lap)

In [10]:
pit_indicator

,ts,value,LapNum
0,14.6425,0,0
1,1624.2000,1,14
2,1684.5000,0,15


In [11]:
def detect_pit_stop(times):
    '''
    See if driver did a pitstop during a lap    
    '''
    # I notice that the times are estimated in the nearest factor of .05 

    # is it possible for me to remove 

In [12]:
fuel_level

,fuel_level,ts,LapNum
0,92.000000,0.00,0
1,92.000000,0.05,0
2,92.000000,0.10,0
3,92.000000,0.15,0
4,92.000000,0.20,0
...,...,...,...
50903,16.962720,2545.15,22
50904,16.962633,2545.20,22
50905,16.962545,2545.25,22
50906,16.962456,2545.30,22


In [13]:
fuel_level[fuel_level['ts'] == 1624.20]

,fuel_level,ts,LapNum
32484,3.727871,1624.2,14


In [14]:
# Create an array of the laps done in the race to iterate through a loop
num_laps = fuel_level['LapNum'].unique()

# Create new dataframe containing fuel usage
fuel_usage = pd.DataFrame(columns=['lapnum', 'laptime(s)', 'fuelused'])

for lap in num_laps:
    # isolate the lap
    timed_lap = fuel_level[fuel_level['LapNum'] == lap]
    
    # Calculate laptime
    laptime = timed_lap['ts'].iloc[-1] - timed_lap['ts'].iloc[0]
    
    # Calculate fuel used
    fuel_used = timed_lap['fuel_level'].iloc[0] - timed_lap['fuel_level'].iloc[-1]

    # save into new dataframe
    row_list = [lap, laptime, fuel_used]
    fuel_usage.loc[len(fuel_usage)] = row_list

In [15]:
fuel_usage

,lapnum,laptime(s),fuelused
0,0.0,14.60,0.000000
1,1.0,223.90,8.259010
2,2.0,107.20,6.289635
3,3.0,107.10,6.383515
4,4.0,106.90,6.292960
5,5.0,105.90,6.365580
6,6.0,107.15,6.213995
7,7.0,106.35,6.162579
8,8.0,106.25,6.290596
9,9.0,108.10,6.395820


In [16]:
# look at lap 13 wher I pitted
fuel_level[fuel_level['LapNum'] == 14]

,fuel_level,ts,LapNum
30406,8.592249,1520.30,14
30407,8.592249,1520.35,14
30408,8.592249,1520.40,14
30409,8.592249,1520.45,14
30410,8.592249,1520.50,14
...,...,...,...
33523,67.434460,1676.15,14
33524,67.432976,1676.20,14
33525,67.432820,1676.25,14
33526,67.432820,1676.30,14


In [17]:
fuel_usage.describe()

,lapnum,laptime(s),fuelused
count,23.00000,23.000000,23.000000
mean,11.00000,110.619565,3.262506
std,6.78233,33.387568,13.610502
min,0.00000,14.600000,-58.840571
25%,5.50000,106.300000,6.143899
50%,11.00000,107.000000,6.292960
75%,16.50000,107.650000,6.376159
max,22.00000,223.900000,8.259010


In [33]:
# remove lap where we pit for now 
fuel_usage[(np.abs(stats.zscore(fuel_usage)) < 3).all(axis=1)].drop(0).describe()

,lapnum,laptime(s),fuelused
count,20.000000,20.000000,20.000000
mean,11.900000,107.485000,6.280960
std,6.348643,3.335657,0.203188
min,2.000000,100.850000,5.693933
25%,6.750000,106.325000,6.160831
50%,11.500000,106.975000,6.302508
75%,17.250000,107.200000,6.372480
max,22.000000,118.250000,6.773835


### Coach dave has already come up with a formula, can use that as a base

FR = (TR / TL) x FL + (2 x FL)

### Where: 
1. FR = Fuel for race (in liters)
2. TR = Race duration in seconds
3. FL = Fuel per lap
4. TL = Average lap time

In [ ]:
# I need to look at the fuel calculations to see what's calculated
'''
1. Calculates the amount of laps estimated based on usage
2. Calculates the time it would take to empty the tank
3. Calculates the fuel used. 

I need to flag stints to be able to calculate estimated fuel left vs actual fuel used
I can then use that to calculate a percentage that shows how close your fuel saving is to optimum 
'''
# Now I would need to calculate the time it would take to empty a tank 
fuel_usage

,lapnum,laptime(s),fuelused
0,0.0,14.60,0.000000
1,1.0,223.90,8.259010
2,2.0,107.20,6.289635
3,3.0,107.10,6.383515
4,4.0,106.90,6.292960
5,5.0,105.90,6.365580
6,6.0,107.15,6.213995
7,7.0,106.35,6.162579
8,8.0,106.25,6.290596
9,9.0,108.10,6.395820


### Below is testing code for a test program. 

The idea is to grab the live data from Le Mans Ultimate and turn it into a program. This will give me more exposure to APIs as well as working with data. Perfect project to keep my skills fresh

In [52]:
import requests
import time

In [82]:
inputs.json()

{'liveInputs': {'di': [{'minmax': {'brakes': {'max': -0.0, 'min': 1.0},
     'clutch': {'max': 0.0, 'min': 0.0},
     'handbrake': {'max': 0.0, 'min': 0.0},
     'handfrontbrake': {'max': 0.0, 'min': 0.0},
     'steerLeft': {'max': -0.0, 'min': 0.5},
     'steerRight': {'max': 1.0, 'min': 0.5},
     'throttle': {'max': -0.0, 'min': 1.0}},
    'raw inputs': {'brakes': 1.0,
     'clutch': 0.0,
     'directManualShift': 0,
     'handbrake': 0.0,
     'handfrontbrake': 0.0,
     'launchControl': False,
     'shiftDown': False,
     'shiftToNeutral': False,
     'shiftUp': False,
     'steerLeft': 0.49868008494377136,
     'steerRight': 0.49868008494377136,
     'tcOverride': False,
     'throttle': 0.2880750596523285}},
   {'minmax': {'brakes': {'max': 0.0, 'min': 0.0},
     'clutch': {'max': 0.0, 'min': 0.0},
     'handbrake': {'max': 0.0, 'min': 0.0},
     'handfrontbrake': {'max': 0.0, 'min': 0.0},
     'steerLeft': {'max': 0.0, 'min': 0.0},
     'steerRight': {'max': 0.0, 'min': 0.0},


In [89]:
# Now let's try this do a try loop and that returns the json values until I press the kkeyboard
try:
    while True:
        inputs = requests.get("http://localhost:6397/rest/options/liveInputs")
        print(inputs.json())
        time.sleep(.1)
except KeyboardInterrupt:
    pass

{'liveInputs': {'di': [{'minmax': {'brakes': {'max': -0.0, 'min': 1.0}, 'clutch': {'max': 0.0, 'min': 0.0}, 'handbrake': {'max': 0.0, 'min': 0.0}, 'handfrontbrake': {'max': 0.0, 'min': 0.0}, 'steerLeft': {'max': -0.0, 'min': 0.5}, 'steerRight': {'max': 1.0, 'min': 0.5}, 'throttle': {'max': -0.0, 'min': 1.0}}, 'raw inputs': {'brakes': 1.0, 'clutch': 0.0, 'directManualShift': 0, 'handbrake': 0.0, 'handfrontbrake': 0.0, 'launchControl': False, 'shiftDown': False, 'shiftToNeutral': False, 'shiftUp': False, 'steerLeft': 0.5019302368164062, 'steerRight': 0.5019302368164062, 'tcOverride': False, 'throttle': 1.0}}, {'minmax': {'brakes': {'max': 0.0, 'min': 0.0}, 'clutch': {'max': 0.0, 'min': 0.0}, 'handbrake': {'max': 0.0, 'min': 0.0}, 'handfrontbrake': {'max': 0.0, 'min': 0.0}, 'steerLeft': {'max': 0.0, 'min': 0.0}, 'steerRight': {'max': 0.0, 'min': 0.0}, 'throttle': {'max': 0.0, 'min': 0.0}}, 'raw inputs': {'brakes': 0.0, 'clutch': 0.0, 'directManualShift': 0, 'handbrake': 0.0, 'handfrontbra

In [ ]:
pd.DataFrame.from_dict(inputs.json())

ValueError: Mixing dicts with non-Series may lead to ambiguous ordering.

In [125]:
test = inputs.json()
test2 = test['liveInputs']['di'][0]['raw inputs']

# This would allow me to see the direct raw inputs of the main controller (in this case, a wheelbase.)
pd.DataFrame.from_dict(test2, orient='index').T

,brakes,clutch,directManualShift,handbrake,handfrontbrake,launchControl,shiftDown,shiftToNeutral,shiftUp,steerLeft,steerRight,tcOverride,throttle
0,1.0,0.0,0,0.0,0.0,False,False,False,False,0.501915,0.501915,False,1.0


In [86]:
requests.post("http://localhost:6397/rest/chat/", data={"message": "Though when I send it through it comes out broken"})

<Response [200]>

In [87]:
try:
    while True:
        requests.post("http://localhost:6397/rest/chat/", data={"message": "OG Ricky Bobby OMG"})
        time.sleep(2)
except KeyboardInterrupt:
    pass

In [126]:
inputs = requests.get("http://localhost:6397/rest/options/liveInputs").json()

In [112]:
test['liveInputs']['di'][0]['raw inputs']

{'brakes': 1.0,
 'clutch': 0.0,
 'directManualShift': 0,
 'handbrake': 0.0,
 'handfrontbrake': 0.0,
 'launchControl': False,
 'shiftDown': False,
 'shiftToNeutral': False,
 'shiftUp': False,
 'steerLeft': 0.5019149780273438,
 'steerRight': 0.5019149780273438,
 'tcOverride': False,
 'throttle': 1.0}

In [ ]:
# Try a function that would create the data frame from logging
def log_driving():
    # Grab the first row
    first_input = requests.get("http://localhost:6397/rest/options/liveInputs").json()
    # convert to dataframe
    input_logging = pd.DataFrame().from_dict(first_input['liveInputs']['di'][0]['raw inputs'], orient='index').T

    # now we log the data 
    try:
        while True:
            # Grab new input
            new_input = requests.get("http://localhost:6397/rest/options/liveInputs").json()

            # append new input to dataframe
            new_row = pd.DataFrame().from_dict(new_input['liveInputs']['di'][0]['raw inputs'], orient='index').T
            input_logging = pd.concat([input_logging, new_row], ignore_index=True)
            #input_logging.reset_index(inplace=True)
            time.sleep(.05)
    except ArithmeticError:
        pass

In [145]:
# Grab the first row
first_input = requests.get("http://localhost:6397/rest/options/liveInputs").json()
# convert to dataframe
input_logging = pd.DataFrame().from_dict(first_input['liveInputs']['di'][0]['raw inputs'], orient='index').T

# now we log the data 
try:
    while True:
        # Grab new input
        new_input = requests.get("http://localhost:6397/rest/options/liveInputs").json()

        # append new input to dataframe
        new_row = pd.DataFrame().from_dict(new_input['liveInputs']['di'][0]['raw inputs'], orient='index').T
        input_logging = pd.concat([input_logging, new_row], ignore_index=True)
        #input_logging.reset_index(inplace=True)
        time.sleep(.05)
except ArithmeticError:
    pass

KeyboardInterrupt: 

In [147]:
input_logging[input_logging['shiftDown'] == True]

,brakes,clutch,directManualShift,handbrake,handfrontbrake,launchControl,shiftDown,shiftToNeutral,shiftUp,steerLeft,steerRight,tcOverride,throttle
524,0.362295,0.0,0,0.0,0.0,False,True,False,False,0.500877,0.500877,False,1.0
525,0.336904,0.0,0,0.0,0.0,False,True,False,False,0.50193,0.50193,False,1.0
526,0.26659,0.0,0,0.0,0.0,False,True,False,False,0.501747,0.501747,False,1.0
527,0.208988,0.0,0,0.0,0.0,False,True,False,False,0.501244,0.501244,False,1.0
533,0.0,0.0,0,0.0,0.0,False,True,False,False,0.502403,0.502403,False,1.0
534,0.0,0.0,0,0.0,0.0,False,True,False,False,0.502007,0.502007,False,1.0
535,0.0,0.0,0,0.0,0.0,False,True,False,False,0.50222,0.50222,False,1.0
536,0.0,0.0,0,0.0,0.0,False,True,False,False,0.502617,0.502617,False,1.0
544,0.324208,0.0,0,0.0,0.0,False,True,False,False,0.502693,0.502693,False,1.0
545,0.414054,0.0,0,0.0,0.0,False,True,False,False,0.501411,0.501411,False,1.0
